# Hello World
## Apache Zeppelin

This note runs in **Apache Zeppelin** on the cluster (for example HDInsight). It reads **HVAC** sample data from `/HdiSamples/`, explores it with Spark **DataFrames**, registers a **temp view**, and runs **SQL** through `spark.sql`. A short overview explains how Spark is organised here; then you **inspect your session** before the imports and CSV read.

## How Spark works here (reminder)

**Apache Zeppelin** sends these paragraphs to Spark through **Livy**. The **driver** coordinates the job; **executors** on worker machines run **tasks** in parallel on **partitions** of the data.

**Data is partitioned.** Large files are read in **chunks**; each chunk can be processed on a different executor. Storage on the cluster is typically under paths such as `/HdiSamples/`.

An **RDD** (**resilient distributed dataset**) is simply a dataset **split into partitions** across the cluster so work can run **in parallel** on many machines.

**Lazy evaluation:** when you call `read.csv` or build a query, Spark often records **what** to do first; heavy work runs when you call an **action** (for example `count()`, `show()`, or `spark.sql(...).show()`).

## Inspect your Spark session

The **master** URL shows how Spark is deployed on this cluster (often `yarn` on HDInsight). **App name** helps you find this job in the Spark UI.

In [ ]:
%livy2.pyspark
print("Spark version:", spark.version)
sc = spark.sparkContext
print("Master:", sc.master)
print("App name:", sc.appName)


### Imports

PySpark and Spark SQL modules. The **%livy2.pyspark** interpreter runs this paragraph on the cluster's Spark session.

In [ ]:
%livy2.pyspark
from pyspark.sql import *
from pyspark.sql.types import *

**Read the HVAC CSV** from the cluster path into a Spark DataFrame. First row as header, schema inferred.

In [ ]:
%livy2.pyspark
csvFile = spark.read.csv('/HdiSamples/HdiSamples/SensorSampleData/hvac/HVAC.csv', header=True, inferSchema=True)


**Row count:** how many records in the DataFrame.

In [ ]:
%livy2.pyspark
# How many rows in dataframe?
csvFile.count()

### HVAC dataset description

The **HVAC** (Heating, Ventilation, and Air Conditioning) sample data contains sensor readings from buildings: date and time, target and actual temperatures, system identifier, system age, and building ID. Below we explore the schema and a sample of the data before registering a temp view for SQL.

In [ ]:
%livy2.pyspark
# Show the schema: column names and types inferred by Spark
csvFile.printSchema()

In [ ]:
%livy2.pyspark
# Preview the first 10 rows
csvFile.show(10)

In [ ]:
%livy2.pyspark
# Summary statistics for numeric columns
csvFile.describe().show()

### Register the DataFrame as a SQL temp view

We register `csvFile` as a temporary view so we can query it with **`spark.sql("...").show()`** in the following paragraphs. That uses the same SQL engine as the DataFrame API.

In [ ]:
%livy2.pyspark
csvFile.createOrReplaceTempView("hvac_ss01shh")

**SQL in this note:** Each query is a normal Python line inside a **`%livy2.pyspark`** paragraph: `spark.sql("...").show()`. That is the same API as in the other course notebooks; there is no separate SQL-only paragraph type required here.

**Verify row count** with SQL (should match the DataFrame count above).

In [ ]:
%livy2.pyspark
spark.sql("SELECT count(*) from hvac_ss01shh").show()

**Describe the table** to see column names and types.

In [ ]:
%livy2.pyspark
spark.sql("DESCRIBE TABLE hvac_ss01shh").show()

**Example query:** temperature difference (target minus actual) for a given date. Positive = heating; negative = cooling.

In [ ]:
%livy2.pyspark
spark.sql("SELECT `buildingID`, (targettemp - actualtemp) AS `temp_diff`, date FROM hvac_ss01shh WHERE date = '6/1/13'").show()